Ячейка 1 — установка

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


Ячейка 2 — импорты

In [ ]:
import os
import json
import zipfile
import requests
import pandas as pd

from pipeline import run_pipeline, normalize_timestamp, UNIFIED_COLUMNS
from parsers.json_parser import parse_json_line
from parsers.syslog_parser import parse_syslog_5424
from parsers.evtx_parser import parse_evtx

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
print("OK")

OK


Ячейка 3 — скачивание данных

In [3]:
EVTX_URL = (
    "https://github.com/sbousseaden/EVTX-ATTACK-SAMPLES/raw/master/"
    "Execution/Exec_sysmon_meterpreter_reversetcp_msipackage.evtx"
)
if not os.path.exists("sample.evtx"):
    r = requests.get(EVTX_URL, timeout=60); r.raise_for_status()
    open("sample.evtx", "wb").write(r.content)
print("EVTX:", os.path.getsize("sample.evtx"), "байт")

CLOUDTRAIL_URL = (
    "https://media.githubusercontent.com/media/splunk/attack_data/master/"
    "datasets/attack_techniques/T1110.003/aws_multiple_login_fail_per_user/cloudtrail.json"
)
if not os.path.exists("cloudtrail.json"):
    r = requests.get(CLOUDTRAIL_URL, timeout=60); r.raise_for_status()
    open("cloudtrail.json", "w", encoding="utf-8").write(r.text)
print("CloudTrail:", os.path.getsize("cloudtrail.json"), "байт")

EVTX: 69632 байт
CloudTrail: 26292 байт


Ячейка 4 — прогон пайплайна на трёх источниках

In [4]:
for path, kind in [
    ("cloudtrail.json", "json"),
    ("sample.evtx", "evtx"),
]:
    df = run_pipeline(path, kind)
    print(f"\n=== {kind} ===")
    print("shape:", df.shape)
    print(df.dtypes)
    display(df.head(3))

# Syslog — из демо-строк
sample_syslog = [
    "<34>1 2025-01-02T03:04:05.123Z host1 sshd 1234 ID47 [x] Failed password for alice",
    "<34>1 2025-01-02T03:04:06.200Z host1 sshd 1235 ID48 [x] Failed password for alice",
    "битая строка",
]
open("sample_syslog.log", "w", encoding="utf-8").write("\n".join(sample_syslog))
df_sys = run_pipeline("sample_syslog.log", "syslog")
print("\n=== syslog ===")
display(df_sys)

[json] разобрано 28, пропущено 0

=== json ===
shape: (28, 6)
timestamp     datetime64[ns, UTC]
source                     object
user                       object
event_type                 object
severity                   object
raw                        object
dtype: object


,timestamp,source,user,event_type,severity,raw
0,2022-10-19 20:33:38+00:00,json,HIDDEN_DUE_TO_SECURITY_REASONS,ConsoleLogin,,"{""eventVersion"": ""1.08"", ""userIdentity"": {""typ..."
1,2022-10-19 20:33:37+00:00,json,HIDDEN_DUE_TO_SECURITY_REASONS,ConsoleLogin,,"{""eventVersion"": ""1.08"", ""userIdentity"": {""typ..."
2,2022-10-19 20:33:37+00:00,json,HIDDEN_DUE_TO_SECURITY_REASONS,ConsoleLogin,,"{""eventVersion"": ""1.08"", ""userIdentity"": {""typ..."



=== evtx ===
shape: (8, 6)
timestamp     datetime64[ns, UTC]
source                     object
user                       object
event_type                 object
severity                   object
raw                        object
dtype: object


,timestamp,source,user,event_type,severity,raw
0,2019-04-30 22:48:58.901474+00:00,evtx,,EventID_2,,"<Event xmlns=""http://schemas.microsoft.com/win..."
1,2019-04-30 22:48:59.260851+00:00,evtx,,EventID_1,,"<Event xmlns=""http://schemas.microsoft.com/win..."
2,2019-04-30 22:49:08.760851+00:00,evtx,,EventID_2,,"<Event xmlns=""http://schemas.microsoft.com/win..."


[syslog] разобрано 2, пропущено 1

=== syslog ===


,timestamp,source,user,event_type,severity,raw
0,2025-01-02 03:04:05.123000+00:00,syslog,,sshd,34,<34>1 2025-01-02T03:04:05.123Z host1 sshd 1234...
1,2025-01-02 03:04:06.200000+00:00,syslog,,sshd,34,<34>1 2025-01-02T03:04:06.200Z host1 sshd 1235...


Ячейка 5 — контрольный пример + тесты

In [5]:
# Контроль: 25 failed с 10.0.0.99 в demo_auth_log.csv (генерируется в лекции)
# Если демо-CSV нет — пропускаем; иначе:
if os.path.exists("demo_auth_log.csv"):
    check = pd.read_csv("demo_auth_log.csv", parse_dates=["timestamp"])
    n = len(check[(check["src_ip"] == "10.0.0.99") &
                  (check["event_type"] == "login_failed")])
    assert n == 25, f"ожидалось 25, получено {n}"
    print("Контрольный пример пройден ✓")

# Юнит-тесты
!pytest tests/ -v

============================= test session starts =============================
platform win32 -- Python 3.12.6, pytest-8.3.4, pluggy-1.7.0 -- c:\Users\danji\OneDrive\Рабочий стол\SWAG\education\po_analyze\Module1\log-pipeline\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\danji\OneDrive\Рабочий стол\SWAG\education\po_analyze\Module1\log-pipeline
plugins: platformdirs-4.14.0
collecting ... collected 18 items

tests/test_evtx.py::test_evtx_importable PASSED                          [  5%]
tests/test_evtx.py::test_evtx_columns_contract PASSED                    [ 11%]
tests/test_evtx.py::test_evtx_missing_file_raises PASSED                 [ 16%]
tests/test_evtx.py::test_evtx_real_file PASSED                           [ 22%]
tests/test_json.py::test_json_ok_basic PASSED                            [ 27%]
tests/test_json.py::test_json_ok_cloudtrail_nested_user PASSED           [ 33%]
tests/test_json.py::test_json_broken PASSED                              [ 38%]
tests/t